In [1]:
import pybullet as p

try:
    p.disconnect()
except:
    pass

### There have been 60 versions in total. (PPO architecture)
### But the best one was V41.

In [ ]:
# V41
# =============================================
# Definition of Environment and Reward Function
# =============================================

import numpy as np
import pybullet as p
import pybullet_data
import gymnasium as gym
from gymnasium import spaces
import math
import platform

class BalanceBotEnv(gym.Env):
    def __init__(self, render=False, phase=3):
        super(BalanceBotEnv, self).__init__()
        self.render_mode = render
        self.phase = phase
        
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        # Retain the 9D observation space from V25
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(9,), dtype=np.float32)

        self.client = p.connect(p.GUI if render else p.DIRECT, options="--width=560 --height=700")
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        
        
        self.max_torque = 25.0
        self.last_action = np.zeros(2)
        self.max_episode_steps = 5000
        self.current_step = 0
        
        self.filtered_v = 0.0
        self.v_error_integral = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetSimulation(physicsClientId=self.client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.client)
        p.loadURDF("plane.urdf", physicsClientId=self.client)
        self.robotId = p.loadURDF("balance_bot.urdf", [0, 0, 0.1], physicsClientId=self.client)
        
        for j in range(p.getNumJoints(self.robotId, physicsClientId=self.client)):
            p.setJointMotorControl2(self.robotId, j, p.VELOCITY_CONTROL, force=0, physicsClientId=self.client)
        
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        start_yaw = p.getEulerFromQuaternion(orn)[2]

        if self.phase == 1:
            self.target_v = 0.0
            self.target_yaw = start_yaw
        elif self.phase == 2:
            self.target_v = self.np_random.uniform(0.1, 0.25)
            self.target_yaw = start_yaw
        else:
            self.target_v = self.np_random.uniform(0.1, 0.25)
            self.target_yaw = self.np_random.uniform(0.6*np.pi, np.pi)
        
        self.last_action = np.zeros(2)
        self.current_step = 0
        self.filtered_v = 0.0
        self.v_error_integral = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        euler = p.getEulerFromQuaternion(orn)
        linear_vel, angular_vel = p.getBaseVelocity(self.robotId, physicsClientId=self.client)
        
        yaw = euler[2]
        raw_v_forward = linear_vel[0] * math.cos(yaw) + linear_vel[1] * math.sin(yaw)
        
        # Physical noise reduction: Filter high-frequency friction vibrations
        self.filtered_v = 0.1 * raw_v_forward + 0.9 * self.filtered_v
        
        v_err = self.target_v - self.filtered_v
        
        # [V41 Pure Integrator] No leaks! Completely eliminate the final 0.05 speed error!
        self.v_error_integral += v_err * (1/240)
        self.v_error_integral = np.clip(self.v_error_integral, -2.0, 2.0)
        
        yaw_err = self.target_yaw - euler[2]
        yaw_err = math.atan2(math.sin(yaw_err), math.cos(yaw_err))
        
        return np.concatenate([
            [euler[1], angular_vel[1], self.filtered_v, v_err, self.v_error_integral, yaw_err, angular_vel[2]],
            self.last_action
        ]).astype(np.float32)

    def step(self, action):
        throttle = action[0]
        steering = action[1] * 0.5
        
        force_l = np.clip((throttle - steering) * self.max_torque, -self.max_torque, self.max_torque)
        force_r = np.clip((throttle + steering) * self.max_torque, -self.max_torque, self.max_torque)

        p.setJointMotorControlArray(self.robotId, [0, 1], p.TORQUE_CONTROL,
                                    forces=[force_l, force_r],
                                    physicsClientId=self.client)
        p.stepSimulation(physicsClientId=self.client)
        self.current_step += 1
        
        obs = self._get_obs()
        pitch, pitch_rate, v_forward, v_err, v_int, yaw_err, yaw_rate = obs[0], obs[1], obs[2], obs[3], obs[4], obs[5], obs[6]
        
        v_z = p.getBaseVelocity(self.robotId, physicsClientId=self.client)[0][2]
        terminated = abs(pitch) > 0.45
        truncated = self.current_step >= self.max_episode_steps

        # ==========================================
        #V41: Ultimate stable form breaking the integral paradox
        # ==========================================
        reward = 15.0
        
        # 1. Anti-rollover soft wall: penalty-free within 0.15, severe quadratic penalty beyond
        excess_pitch = max(0.0, abs(pitch) - 0.15)
        reward -= 1500.0 * (excess_pitch ** 2)

        # 2. Pseudo-absolute value smoothing: erases zero-point jitter, firmly locks the target
        eps = 0.0001
        sm_v_err = math.sqrt(v_err**2 + eps)
        sm_yaw_err = math.sqrt(yaw_err**2 + eps)

        # 3. Inherits core tracking pull (Removed the erroneous bonus that kept the integrator at 0)
        reward += 20.0 * math.exp(-sm_v_err / 0.1)
        reward += 10.0 * math.cos(yaw_err)
        reward += 15.0 * math.exp(-sm_yaw_err / 0.15)

        # 4. Release straight-line handbrake: 0 penalty for straight line to unleash top speed, heavy penalty for large angles to prevent taking off
        reward -= 2.0 * abs(v_forward) * (yaw_err ** 2)

        # 5. Verified perfect physical damping sweet spot
        reward -= 5.0 * np.sum(np.square(action - self.last_action))
        reward -= 1.0 * np.sum(np.square(action))
        reward -= 10.0 * abs(v_z)      
        reward -= 1.0 * abs(pitch_rate)
        reward -= 2.0 * abs(yaw_rate)
        
        if terminated: reward = -2000.0

        self.last_action = action.copy()
        return obs, reward, terminated, truncated, {}

    def close(self):
        p.disconnect(physicsClientId=self.client)

In [ ]:
# V41 
# ==========================================
# PPO Full Curriculum Training Pipeline
# ==========================================

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import os

steps_p1 = 400000  
steps_p2 = 600000  
steps_p3 = 1000000  

def make_env_phase(phase_level):
    def _init():
        return Monitor(BalanceBotEnv(render=False, phase=phase_level))
    return _init

print(f"=========== Starting V41 (Paradox-breaking Pure Tracking Version) Curriculum Learning ===========")

print(f"\n>>> Phase 1/3: Training Absolute Stationary Balance...")
env_p1 = DummyVecEnv([make_env_phase(1)])
env_p1 = VecNormalize(env_p1, norm_obs=True, norm_reward=False, clip_obs=10.)
model_p1 = PPO("MlpPolicy", env_p1, verbose=1, learning_rate=1e-4,
               n_steps=4096, batch_size=256, n_epochs=15, ent_coef=0.01)
model_p1.learn(total_timesteps=steps_p1)
model_p1.save("bot_v41_phase1")
env_p1.save("vec_norm_v41_p1.pkl")

print(f"\n>>> Phase 2/3: Training Straight-Line Tracking...")
env_p2 = DummyVecEnv([make_env_phase(2)])
env_p2 = VecNormalize.load("vec_norm_v41_p1.pkl", env_p2)
env_p2.training = True
model_p2 = PPO.load("bot_v41_phase1", env=env_p2)
model_p2.learning_rate = 5e-5
model_p2.learn(total_timesteps=steps_p2, reset_num_timesteps=False)
model_p2.save("bot_v41_phase2")
env_p2.save("vec_norm_v41_p2.pkl")

print(f"\n>>> Phase 3/3: Training Omnidirectional Cruising...")
env_p3 = DummyVecEnv([make_env_phase(3)])
env_p3 = VecNormalize.load("vec_norm_v41_p2.pkl", env_p3)
env_p3.training = True
model_p3 = PPO.load("bot_v41_phase2", env=env_p3)
model_p3.learning_rate = 2e-5
model_p3.ent_coef = 0.005    
model_p3.learn(total_timesteps=steps_p3, reset_num_timesteps=False)
model_p3.save("balance_bot_v41_curriculum")
env_p3.save("vec_normalize_v41.pkl")

print(f"\n=========== Congratulations, V41 Graduated! ===========")

In [ ]:
# V41
# ==============================
# Model Evaluation
# ==============================
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO

eval_env_raw = DummyVecEnv([lambda: Monitor(BalanceBotEnv(render=False, phase=3))])
eval_env = VecNormalize.load("vec_normalize_v41.pkl", eval_env_raw)
eval_env.training = False
eval_env.norm_reward = False

model = PPO.load("balance_bot_v41_curriculum")

print("Executing V41 Performance Evaluation...")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Evaluation Complete! V41 Final Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
# V41
# =============================
# Rendered Demonstration
# =============================
import time
import math
import pybullet as p
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3 import PPO

try: p.disconnect()
except: pass

test_env_raw = DummyVecEnv([lambda: BalanceBotEnv(render=True, phase=3)])
test_env = VecNormalize.load("vec_normalize_v41.pkl", test_env_raw)
test_env.training = False

model = PPO.load("balance_bot_v41_curriculum")
obs = test_env.reset()

robot_id = test_env.get_attr("robotId")[0]
client_id = test_env.get_attr("client")[0]
v_target = test_env.get_attr("target_v")[0]
yaw_target = test_env.get_attr("target_yaw")[0]

p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=client_id)
p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30,
                             cameraTargetPosition=[0, 0, 0.2], physicsClientId=client_id)

print(f"\n" + "="*50)
print(f">>> V41 Graduation Road Test | Target Speed: {v_target:.2f} m/s | Target Yaw: {yaw_target:.2f} rad")
print("="*50 + "\n")

try:
    for i in range(2400):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = test_env.step(action)
        
        pos, orn = p.getBasePositionAndOrientation(robot_id, physicsClientId=client_id)
        lin_vel, _ = p.getBaseVelocity(robot_id, physicsClientId=client_id)
        yaw_now = p.getEulerFromQuaternion(orn)[2]
        
        p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30,
                                     cameraTargetPosition=[pos[0], pos[1], 0.2], physicsClientId=client_id)
        
        if i % 120 == 0:
            v_now = lin_vel[0] * math.cos(yaw_now) + lin_vel[1] * math.sin(yaw_now)
            yaw_error_deg = math.degrees(math.atan2(math.sin(yaw_target-yaw_now), math.cos(yaw_target-yaw_now)))
            
            if abs(yaw_error_deg) > 15: mode = "🔄 Cornering (Anti-Wheelie)"
            elif abs(v_target - v_now) > 0.05: mode = "⚡ Full Speed Pursuit"
            else: mode = "🎯 Steady State Deadlock"
                
            print(f"Step {i:4d} | Speed: {v_now:>5.2f} / {v_target:>5.2f} | Yaw Error: {yaw_error_deg:>6.1f}° | Status: {mode}")
            
        time.sleep(1/240)

except KeyboardInterrupt: pass
finally:
    print("Cleaning up physics engine...")
    test_env.close()
    try: p.disconnect(physicsClientId=client_id)
    except: pass
    print("Environment closed.")

In [ ]:
# ===========================================
# V41 - Ultimate Disturbance Video Demo 
# ===========================================
import time
import math
import pybullet as p
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3 import PPO

try: p.disconnect()
except: pass

# ---  (Director's Console) ---
KICK_FORCE_1 = 600      
KICK_FORCE_2 = 200
KICK_HEIGHT = 0.15    
CRUISE_SPEED = 0.24   
# ---------------------------------------------

# 1. Load Phase 2
test_env_raw = DummyVecEnv([lambda: BalanceBotEnv(render=True, phase=2)])
test_env = VecNormalize.load("vec_normalize_v41.pkl", test_env_raw)
test_env.training = False

model = PPO.load("balance_bot_v41_curriculum")
obs = test_env.reset()

# 2. Set Target
underlying_env = test_env.envs[0].unwrapped
underlying_env.target_v = CRUISE_SPEED
underlying_env.target_yaw = 0.0

robot_id = underlying_env.robotId
client_id = underlying_env.client

p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=client_id)
p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-20,
                             cameraTargetPosition=[0, 0, 0.2], physicsClientId=client_id)

print("\n" + "="*65)
print(f">>> DRL DISTRUBANCE DEMO | Target Vel: {CRUISE_SPEED:.2f} m/s | Target Yaw: 0.00 rad")
print("="*65 + "\n")

try:
    for i in range(2400): 
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = test_env.step(action)
        
        pos, orn = p.getBasePositionAndOrientation(robot_id, physicsClientId=client_id)
        lin_vel, _ = p.getBaseVelocity(robot_id, physicsClientId=client_id)
        yaw_now = p.getEulerFromQuaternion(orn)[2]
        
        # Camera follow
        p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-20,
                                     cameraTargetPosition=[pos[0], pos[1], 0.2], physicsClientId=client_id)
        
        # ==========================================
        # 💥 External Disturbance Injection
        # ==========================================
        if i == 400:
            p.applyExternalForce(objectUniqueId=robot_id, linkIndex=-1,
                                 forceObj=[-KICK_FORCE_1, 0, 0], posObj=[0, 0, KICK_HEIGHT],
                                 flags=p.WORLD_FRAME, physicsClientId=client_id)
            print("\n" + "⚠️"*15)
            print(f"💥 BOOM! [Step 400] FRONTAL PUSH ({KICK_FORCE_1}N @ Z={KICK_HEIGHT}m) INJECTED!")
            print("⚠️"*15 + "\n")

        elif i == 800:
            p.applyExternalForce(objectUniqueId=robot_id, linkIndex=-1,
                                 forceObj=[0, KICK_FORCE_2, 0], posObj=[0, 0, KICK_HEIGHT],
                                 flags=p.WORLD_FRAME, physicsClientId=client_id)
            print("\n" + "⚠️"*15)
            print(f"💥 BOOM! [Step 800] LATERAL KICK ({KICK_FORCE_2}N @ Z={KICK_HEIGHT}m) INJECTED!")
            print("⚠️"*15 + "\n")

        # ==========================================
        # 📊 Telemetry Logging 
        # ==========================================
        if i % 120 == 0:
            v_now = lin_vel[0] * math.cos(yaw_now) + lin_vel[1] * math.sin(yaw_now)
            yaw_error_deg = math.degrees(math.atan2(math.sin(0.0-yaw_now), math.cos(0.0-yaw_now)))
            
            if abs(yaw_error_deg) > 15: mode = "🔄 Cornering (Anti-Wheelie)"
            elif abs(CRUISE_SPEED - v_now) > 0.05: mode = "⚡ Full Speed Pursuit"
            else: mode = "🎯 Steady State Deadlock"
                
            print(f"Step {i:4d} | Speed: {v_now:>5.2f} / {CRUISE_SPEED:>5.2f} | Yaw Error: {yaw_error_deg:>6.1f}° | Status: {mode}")
            
        time.sleep(1/240) 

except KeyboardInterrupt: pass
finally:
    test_env.close()
    try: p.disconnect(physicsClientId=client_id)
    except: pass

### BELOW: SAC Training using V41's reward design

In [ ]:
# ==========================================
# SAC Full Curriculum Training Pipeline
# ==========================================
import os
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Define curriculum step milestones (Total: 2M steps)
steps_p1 = 400000  
steps_p2 = 600000  
steps_p3 = 1000000  

def make_env_phase(phase_level):
    """Factory function to generate monitored environments for specific phases."""
    def _init():
        return Monitor(BalanceBotEnv(render=False, phase=phase_level))
    return _init

print("="*70)
print(">>> Starting SAC (Max Entropy) Curriculum Learning Pipeline")
print("="*70)

# ---------------------------------------------------------
# Phase 1: Absolute Stationary Balance
# ---------------------------------------------------------
print("\n[Phase 1/3] Training Absolute Stationary Balance...")
env_p1 = DummyVecEnv([make_env_phase(1)])
# Initialize observation normalization (crucial for SAC's neural networks)
env_p1 = VecNormalize(env_p1, norm_obs=True, norm_reward=False, clip_obs=10.)

# Initialize SAC with a standard learning rate for rapid initial exploration.
# ent_coef='auto' allows SAC to dynamically adjust its exploration temperature.
model_p1 = SAC("MlpPolicy", env_p1, verbose=1, learning_rate=3e-4)

model_p1.learn(total_timesteps=steps_p1)
model_p1.save("bot_sac_phase1")
env_p1.save("vec_norm_sac_p1.pkl")
print("✅ Phase 1 Complete. Base mechanics acquired.")

# ---------------------------------------------------------
# Phase 2: Straight-Line Tracking
# ---------------------------------------------------------
print("\n[Phase 2/3] Training Straight-Line Tracking...")
env_p2 = DummyVecEnv([make_env_phase(2)])
# Load the normalization statistics from Phase 1
env_p2 = VecNormalize.load("vec_norm_sac_p1.pkl", env_p2)
env_p2.training = True

model_p2 = SAC.load("bot_sac_phase1", env=env_p2)
# Step down the learning rate to refine the tracking policy without forgetting balance
model_p2.learning_rate = 1e-4  
model_p2.learn(total_timesteps=steps_p2, reset_num_timesteps=False)

model_p2.save("bot_sac_phase2")
env_p2.save("vec_norm_sac_p2.pkl")
print("✅ Phase 2 Complete. Velocity tracking refined.")

# ---------------------------------------------------------
# Phase 3: Omnidirectional Cruising (The Ultimate Test)
# ---------------------------------------------------------
print("\n[Phase 3/3] Training Omnidirectional Cruising...")
env_p3 = DummyVecEnv([make_env_phase(3)])
# Load the accumulated normalization statistics
env_p3 = VecNormalize.load("vec_norm_sac_p2.pkl", env_p3)
env_p3.training = True

model_p3 = SAC.load("bot_sac_phase2", env=env_p3)
# Critically low learning rate for Phase 3.
# The 0.45 rad terminal state is highly punitive; aggressive updates here will cause catastrophic forgetting.
model_p3.learning_rate = 5e-5  
model_p3.learn(total_timesteps=steps_p3, reset_num_timesteps=False)

# Save the final champion weights
model_p3.save("balance_bot_sac_curriculum")
env_p3.save("vec_normalize_sac.pkl")

print("\n" + "="*70)
print("🏆 Congratulations! SAC Curriculum Training Graduated!")
print("="*70)

In [ ]:
# ==============================
# SAC Model Evaluation
# ==============================
# The result was poor.

from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

eval_env_raw = DummyVecEnv([lambda: Monitor(BalanceBotEnv(render=False, phase=3))])
eval_env = VecNormalize.load("vec_normalize_sac.pkl", eval_env_raw)
eval_env.training = False
eval_env.norm_reward = False

model = SAC.load("balance_bot_sac_curriculum")

print("Executing SAC Performance Evaluation...")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Evaluation Complete! SAC Final Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

### BELOW: Comparison among PPO, SAC and DDPG, with hyperparameter grid search

In [ ]:
# ==========================================================
# Full Grid Hyperparameter Search: PPO vs SAC vs DDPG
# ==========================================================
import os
import numpy as np
import pickle
import pybullet as p
from stable_baselines3 import PPO, DDPG, SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

# Force disconnect to prevent any ghost physics clients
try: p.disconnect()
except: pass

# ==========================================
# 1. Experiment Configuration
# ==========================================
# 100k timesteps per run is optimal for an overnight 27-run batch.
TOTAL_TIMESTEPS = 100000  
EVAL_FREQ = 5000         
LOG_DIR = "./full_grid_search_logs/"

algorithms = {
    "PPO": PPO,
    "SAC": SAC,
    "DDPG": DDPG
}

# The Hyperparameter Grids (3x3 = 9 combinations per algorithm)
learning_rates = {"High_1e-3": 1e-3, "Base_1e-4": 1e-4, "Low_1e-5": 1e-5}
discount_factors = {"Gamma_0.90": 0.90, "Gamma_0.99": 0.99, "Gamma_0.999": 0.999}

# Unified results dictionary to store all extracted metrics
results = {}

def make_eval_env():
    # Phase 2 (Straight-Line) provides the cleanest signal for learning efficiency
    return Monitor(BalanceBotEnv(render=False, phase=2))

print("="*75)
print(">>> INITIATING OVERNIGHT FULL GRID SEARCH (3x3x3)")
print(f">>> Target: {len(algorithms) * len(learning_rates) * len(discount_factors)} Total Training Runs")
print(f">>> Timesteps per run: {TOTAL_TIMESTEPS}")
print("="*75 + "\n")

# ==========================================
# 2. Automated Nested Training Loop
# ==========================================
for algo_name, AlgoClass in algorithms.items():
    print(f"\n" + "#"*60)
    print(f"🚀 NOW EVALUATING ARCHITECTURE: {algo_name}")
    print("#"*60)
    
    results[algo_name] = {}
    
    # Nested loops for true Grid Search OFAT (One-Factor-At-A-Time) is eliminated
    for lr_name, lr in learning_rates.items():
        for gamma_name, gamma in discount_factors.items():
            
            combo_name = f"{lr_name}_AND_{gamma_name}"
            print(f"\n--- Training [ {algo_name} ] | {combo_name} ---")
            
            log_path = os.path.join(LOG_DIR, algo_name, combo_name)
            os.makedirs(log_path, exist_ok=True)
            
            train_env = DummyVecEnv([make_eval_env])
            eval_env = DummyVecEnv([make_eval_env])
            
            # verbose=1 in Callback prints progress every EVAL_FREQ steps.
            eval_cb = EvalCallback(eval_env, best_model_save_path=log_path,
                                   log_path=log_path, eval_freq=EVAL_FREQ,
                                   deterministic=True, render=False, verbose=1)
            
            # verbose=0 in Model prevents Jupyter from crashing due to massive terminal output.
            # Injecting both LR and Gamma simultaneously for coupling effect analysis
            model = AlgoClass("MlpPolicy", train_env, learning_rate=lr, gamma=gamma, verbose=0)
            
            # Execute Training
            model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_cb)
            
            # Extract evaluation data and store it in memory
            data = np.load(os.path.join(log_path, 'evaluations.npz'))
            results[algo_name][combo_name] = {
                'steps': data['timesteps'],
                'mean': data['results'].mean(axis=1),
                'std': data['results'].std(axis=1),
                'lr': lr,
                'gamma': gamma
            }

# ==========================================
# 3. Data Serialization (Pickle Packaging)
# ==========================================
print("\n" + "="*75)
print(">>> GRID SEARCH COMPLETE! Packaging data...")

save_path = "master_sweep_results.pkl"
with open(save_path, 'wb') as f:
    pickle.dump(results, f)

print(f"🎉 All grid search data has been safely serialized to: {save_path}")
print("You can now safely shut down the kernel. Tomorrow we will load this file to generate the heatmaps!")
print("="*75)

In [ ]:
# ==========================================================
# Data Visualization: Hyperparameter Heatmaps & Best Curves
# ==========================================================
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the overnight grid search data
data_path = "master_sweep_results.pkl"
print(f"Loading data from {data_path}...")
with open(data_path, 'rb') as f:
    results = pickle.load(f)

# Define the grid axes based on our sweep
lr_keys = ["High_1e-3", "Base_1e-4", "Low_1e-5"]
gamma_keys = ["Gamma_0.90", "Gamma_0.99", "Gamma_0.999"]
algorithms = list(results.keys())

print("Data loaded successfully! Generating publication-ready plots...\n")

# ==========================================
# Plot 1: Hyperparameter Performance Heatmaps
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Hyperparameter Coupling Effect: Max Mean Reward (Phase 2)", fontsize=16, fontweight='bold', y=1.05)

best_combos = {} # To store the best combo for Plot 2

for i, algo in enumerate(algorithms):
    # Initialize a 3x3 matrix for the heatmap
    heatmap_data = np.zeros((3, 3))
    best_score = -np.inf
    best_combo_key = ""
    
    for row, lr_k in enumerate(lr_keys):
        for col, gamma_k in enumerate(gamma_keys):
            combo_key = f"{lr_k}_AND_{gamma_k}"
            # Extract the maximum mean reward achieved during this training run
            max_reward = np.max(results[algo][combo_key]['mean'])
            heatmap_data[row, col] = max_reward
            
            # Track the best combo for this algorithm
            if max_reward > best_score:
                best_score = max_reward
                best_combo_key = combo_key
                
    best_combos[algo] = best_combo_key
    
    # Draw the heatmap
    sns.heatmap(heatmap_data, ax=axes[i], annot=True, fmt=".0f", cmap="YlGnBu",
                xticklabels=[g.split('_')[1] for g in gamma_keys],
                yticklabels=[l.split('_')[1] for l in lr_keys],
                cbar_kws={'label': 'Max Mean Reward'} if i == 2 else {"drawedges": False})
    
    axes[i].set_title(f"{algo} (Best: {best_score:.0f})", fontsize=14, fontweight='bold')
    axes[i].set_xlabel("Discount Factor ($\gamma$)")
    if i == 0: axes[i].set_ylabel("Learning Rate ($\\alpha$)")

plt.tight_layout()
plt.savefig("Sweep_Heatmaps.png", dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# Plot 2: Best-in-Class Learning Curves
# ==========================================
plt.figure(figsize=(10, 6))
colors = {'PPO': '#1f77b4', 'SAC': '#2ca02c', 'DDPG': '#ff7f0e'}

for algo, combo_key in best_combos.items():
    data = results[algo][combo_key]
    steps = data['steps']
    mean_rew = data['mean']
    std_rew = data['std']
    
    # Clean up combo name for the legend (e.g., "PPO (1e-4, 0.99)")
    lr_val = data['lr']
    gamma_val = data['gamma']
    label_str = f"{algo} ($\\alpha$={lr_val}, $\gamma$={gamma_val})"
    
    plt.plot(steps, mean_rew, label=label_str, color=colors[algo], linewidth=2.5)
    plt.fill_between(steps, mean_rew - std_rew, mean_rew + std_rew, color=colors[algo], alpha=0.15)

plt.title("Ultimate Sprint Comparison (Using Optimal Hyperparameters)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Training Timesteps", fontsize=14)
plt.ylabel("Evaluation Mean Reward", fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig("Best_Learning_Curves.png", dpi=300, bbox_inches='tight')
plt.show()

print("="*75)
print("✅ Visualization Complete!")
print("- 'Sweep_Heatmaps.png' and 'Best_Learning_Curves.png' have been saved to your folder.")
print("="*75)

### BELOW: Classic faliure examples. 

First run the environment definition code, then run the demo code. 

In [ ]:
'''V36: The unstable oscillation'''
# =============================================
# Environment Definition 
# (Actually this is V41 but it can also be used to load V36)
# =============================================

import numpy as np
import pybullet as p
import pybullet_data
import gymnasium as gym
from gymnasium import spaces
import math
import platform

class BalanceBotEnv(gym.Env):
    def __init__(self, render=False, phase=3):
        super(BalanceBotEnv, self).__init__()
        self.render_mode = render
        self.phase = phase
        
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(9,), dtype=np.float32)

        self.client = p.connect(p.GUI if render else p.DIRECT, options="--width=560 --height=700")
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        
        
        self.max_torque = 25.0
        self.last_action = np.zeros(2)
        self.max_episode_steps = 5000
        self.current_step = 0
        
        self.filtered_v = 0.0
        self.v_error_integral = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetSimulation(physicsClientId=self.client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.client)
        p.loadURDF("plane.urdf", physicsClientId=self.client)
        self.robotId = p.loadURDF("balance_bot.urdf", [0, 0, 0.1], physicsClientId=self.client)
        
        for j in range(p.getNumJoints(self.robotId, physicsClientId=self.client)):
            p.setJointMotorControl2(self.robotId, j, p.VELOCITY_CONTROL, force=0, physicsClientId=self.client)
        
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        start_yaw = p.getEulerFromQuaternion(orn)[2]

        if self.phase == 1:
            self.target_v = 0.0
            self.target_yaw = start_yaw
        elif self.phase == 2:
            self.target_v = self.np_random.uniform(0.1, 0.25)
            self.target_yaw = start_yaw
        else:
            self.target_v = self.np_random.uniform(0.1, 0.25)
            self.target_yaw = self.np_random.uniform(0.6*np.pi, np.pi)
        
        self.last_action = np.zeros(2)
        self.current_step = 0
        self.filtered_v = 0.0
        self.v_error_integral = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        euler = p.getEulerFromQuaternion(orn)
        linear_vel, angular_vel = p.getBaseVelocity(self.robotId, physicsClientId=self.client)
        
        yaw = euler[2]
        raw_v_forward = linear_vel[0] * math.cos(yaw) + linear_vel[1] * math.sin(yaw)
        
        self.filtered_v = 0.1 * raw_v_forward + 0.9 * self.filtered_v
        
        v_err = self.target_v - self.filtered_v
        
        self.v_error_integral += v_err * (1/240)
        self.v_error_integral = np.clip(self.v_error_integral, -2.0, 2.0)
        
        yaw_err = self.target_yaw - euler[2]
        yaw_err = math.atan2(math.sin(yaw_err), math.cos(yaw_err))
        
        return np.concatenate([
            [euler[1], angular_vel[1], self.filtered_v, v_err, self.v_error_integral, yaw_err, angular_vel[2]],
            self.last_action
        ]).astype(np.float32)

    def step(self, action):
        throttle = action[0]
        steering = action[1] * 0.5
        
        force_l = np.clip((throttle - steering) * self.max_torque, -self.max_torque, self.max_torque)
        force_r = np.clip((throttle + steering) * self.max_torque, -self.max_torque, self.max_torque)

        p.setJointMotorControlArray(self.robotId, [0, 1], p.TORQUE_CONTROL,
                                    forces=[force_l, force_r],
                                    physicsClientId=self.client)
        p.stepSimulation(physicsClientId=self.client)
        self.current_step += 1
        
        obs = self._get_obs()
        pitch, pitch_rate, v_forward, v_err, v_int, yaw_err, yaw_rate = obs[0], obs[1], obs[2], obs[3], obs[4], obs[5], obs[6]
        
        v_z = p.getBaseVelocity(self.robotId, physicsClientId=self.client)[0][2]
        terminated = abs(pitch) > 0.45
        truncated = self.current_step >= self.max_episode_steps

        reward = 15.0

        excess_pitch = max(0.0, abs(pitch) - 0.15)
        reward -= 1500.0 * (excess_pitch ** 2)

        eps = 0.0001
        sm_v_err = math.sqrt(v_err**2 + eps)
        sm_yaw_err = math.sqrt(yaw_err**2 + eps)

        reward += 20.0 * math.exp(-sm_v_err / 0.1)
        reward += 10.0 * math.cos(yaw_err)
        reward += 15.0 * math.exp(-sm_yaw_err / 0.15)

        reward -= 2.0 * abs(v_forward) * (yaw_err ** 2)

        reward -= 5.0 * np.sum(np.square(action - self.last_action))
        reward -= 1.0 * np.sum(np.square(action))
        reward -= 10.0 * abs(v_z)      
        reward -= 1.0 * abs(pitch_rate)
        reward -= 2.0 * abs(yaw_rate)
        
        if terminated: reward = -2000.0

        self.last_action = action.copy()
        return obs, reward, terminated, truncated, {}

    def close(self):
        p.disconnect(physicsClientId=self.client)

In [ ]:
'''V36: The unstable oscillation'''
# =============================
# Rendered Demonstration
# =============================
import time
import math
import pybullet as p
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3 import PPO

try: p.disconnect()
except: pass

test_env_raw = DummyVecEnv([lambda: BalanceBotEnv(render=True, phase=3)])
test_env = VecNormalize.load("vec_normalize_v36.pkl", test_env_raw)
test_env.training = False

model = PPO.load("balance_bot_v36_curriculum")
obs = test_env.reset()

robot_id = test_env.get_attr("robotId")[0]
client_id = test_env.get_attr("client")[0]
v_target = test_env.get_attr("target_v")[0]
yaw_target = test_env.get_attr("target_yaw")[0]

p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=client_id)
p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30,
                             cameraTargetPosition=[0, 0, 0.2], physicsClientId=client_id)

print(f"\n" + "="*50)
print(f">>> V41 Graduation Road Test | Target Speed: {v_target:.2f} m/s | Target Yaw: {yaw_target:.2f} rad")
print("="*50 + "\n")

try:
    for i in range(2400):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = test_env.step(action)
        
        pos, orn = p.getBasePositionAndOrientation(robot_id, physicsClientId=client_id)
        lin_vel, _ = p.getBaseVelocity(robot_id, physicsClientId=client_id)
        yaw_now = p.getEulerFromQuaternion(orn)[2]
        
        p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30,
                                     cameraTargetPosition=[pos[0], pos[1], 0.2], physicsClientId=client_id)
        
        if i % 120 == 0:
            v_now = lin_vel[0] * math.cos(yaw_now) + lin_vel[1] * math.sin(yaw_now)
            yaw_error_deg = math.degrees(math.atan2(math.sin(yaw_target-yaw_now), math.cos(yaw_target-yaw_now)))
            
            if abs(yaw_error_deg) > 15: mode = "🔄 Cornering (Anti-Wheelie)"
            elif abs(v_target - v_now) > 0.05: mode = "⚡ Full Speed Pursuit"
            else: mode = "🎯 Steady State Deadlock"
                
            print(f"Step {i:4d} | Speed: {v_now:>5.2f} / {v_target:>5.2f} | Yaw Error: {yaw_error_deg:>6.1f}° | Status: {mode}")
            
        time.sleep(1/240)

except KeyboardInterrupt: pass
finally:
    print("Cleaning up physics engine...")
    test_env.close()
    try: p.disconnect(physicsClientId=client_id)
    except: pass
    print("Environment closed.")

In [ ]:
'''V10: Suicider'''
#======================================
# Environment Definition
#======================================

import numpy as np
import pybullet as p
import pybullet_data
import gymnasium as gym
from gymnasium import spaces
import math

class BalanceBotEnv(gym.Env):
    def __init__(self, render=False):
        super(BalanceBotEnv, self).__init__()
        self.render_mode = render
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32)

        self.client = p.connect(p.GUI if render else p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        
        if render:
            self._setup_visuals()
        
        self.max_torque = 16.0 
        self.last_action = np.zeros(2)
        self.max_episode_steps = 5000
        self.current_step = 0

    def _setup_visuals(self):
        p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=self.client)
        p.resetDebugVisualizerCamera(cameraDistance=1.2, cameraYaw=45, cameraPitch=-30, 
                                   cameraTargetPosition=[0, 0, 0.2], physicsClientId=self.client)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetSimulation(physicsClientId=self.client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.client)
        p.loadURDF("plane.urdf", physicsClientId=self.client)
        self.robotId = p.loadURDF("balance_bot.urdf", [0, 0, 0.1], physicsClientId=self.client)
        
        for j in range(p.getNumJoints(self.robotId, physicsClientId=self.client)):
            p.setJointMotorControl2(self.robotId, j, p.VELOCITY_CONTROL, force=0, physicsClientId=self.client)
        
        self.target_v = self.np_random.uniform(-0.1, 0.6)
        self.target_yaw = self.np_random.uniform(-np.pi, np.pi)
        
        self.last_action = np.zeros(2)
        self.current_step = 0
        return self._get_obs(), {}

    def _get_obs(self):
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        euler = p.getEulerFromQuaternion(orn)
        linear_vel, angular_vel = p.getBaseVelocity(self.robotId, physicsClientId=self.client)
        
        yaw = euler[2]
        v_forward = linear_vel[0] * math.cos(yaw) + linear_vel[1] * math.sin(yaw)
        
        yaw_err = self.target_yaw - euler[2]
        yaw_err = math.atan2(math.sin(yaw_err), math.cos(yaw_err))
        v_err = self.target_v - v_forward
        
        return np.concatenate([
            [euler[1], angular_vel[1], v_forward, v_err, yaw_err, angular_vel[2]],
            self.last_action
        ]).astype(np.float32)

    def step(self, action):
        p.setJointMotorControlArray(self.robotId, [0, 1], p.TORQUE_CONTROL, 
                                    forces=[action[0]*self.max_torque, action[1]*self.max_torque], 
                                    physicsClientId=self.client)
        p.stepSimulation(physicsClientId=self.client)
        self.current_step += 1

        obs = self._get_obs()
        pitch, pitch_rate, v_forward, v_err, yaw_err, yaw_rate = obs[0], obs[1], obs[2], obs[3], obs[4], obs[5]
        v_z = p.getBaseVelocity(self.robotId, physicsClientId=self.client)[0][2]
        
        terminated = abs(pitch) > 0.785
        truncated = self.current_step >= self.max_episode_steps

        reward = 5.0 
        reward += 5.0 * math.cos(pitch) 

        reward -= 25.0 * abs(v_err)  
        reward -= 15.0 * abs(yaw_err)  
        
        reward -= 1.0 * abs(pitch_rate)
        reward -= 5.0 * abs(yaw_rate)
        reward -= 2.0 * np.sum(np.square(action - self.last_action))
        reward -= 5.0 * abs(v_z) 

        if terminated: reward = -1000.0 

        self.last_action = action.copy()
        return obs, reward, terminated, truncated, {}

    def close(self):
        p.disconnect(physicsClientId=self.client)

In [ ]:
'''V10: Suicider'''
#======================================
# Rendered Demonstration
#======================================
import time
try: p.disconnect() 
except: pass

test_env_raw = DummyVecEnv([lambda: BalanceBotEnv(render=True)])
test_env = VecNormalize.load("vec_normalize_v10.pkl", test_env_raw)
test_env.training = False

model = PPO.load("balance_bot_navigate_v10")
obs = test_env.reset()
robot_id = test_env.get_attr("robotId")[0]
v_target = test_env.get_attr("target_v")[0]
yaw_target = test_env.get_attr("target_yaw")[0]

print(f"\nTarget Vel: {v_target:.2f} | Target Yaw: {yaw_target:.2f}\n")

try:
    for i in range(2400):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = test_env.step(action)
        
        if i % 120 == 0:
            lin_vel, _ = p.getBaseVelocity(robot_id)
            _, orn = p.getBasePositionAndOrientation(robot_id)
            yaw = p.getEulerFromQuaternion(orn)[2]
            v_now = lin_vel[0] * math.cos(yaw) + lin_vel[1] * math.sin(yaw)
            
            print(f"Step {i:4d} | Actual Vel: {v_now:5.2f} | Yaw Error: {math.sin(yaw_target-yaw):5.2f}")
        time.sleep(1/240)
except KeyboardInterrupt: pass
finally: test_env.close()

In [ ]:
'''V7: Endless Gyro'''
#======================================
# Environment Definition
#======================================
import numpy as np
import pybullet as p
import pybullet_data
import gymnasium as gym
from gymnasium import spaces
import math

class BalanceBotEnv(gym.Env):
    def __init__(self, render=False):
        super(BalanceBotEnv, self).__init__()
        self.render_mode = render
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32)

        self.client = p.connect(p.GUI if render else p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        
        if render:
            self._setup_visuals()
        
        self.max_torque = 20.0 
        self.last_action = np.zeros(2)
        self.last_v = 0.0
        
        self.max_episode_steps = 5000  
        self.current_step = 0

    def _setup_visuals(self):
        p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=self.client)
        p.resetDebugVisualizerCamera(cameraDistance=1.2, cameraYaw=45, cameraPitch=-30, 
                                   cameraTargetPosition=[0, 0, 0.2], physicsClientId=self.client)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetSimulation(physicsClientId=self.client)
        p.setGravity(0, 0, -9.81, physicsClientId=self.client)
        p.loadURDF("plane.urdf", physicsClientId=self.client)
        self.robotId = p.loadURDF("balance_bot.urdf", [0, 0, 0.1], physicsClientId=self.client)
        
        for j in range(p.getNumJoints(self.robotId, physicsClientId=self.client)):
            p.setJointMotorControl2(self.robotId, j, p.VELOCITY_CONTROL, force=0, physicsClientId=self.client)
        
        self.target_v = self.np_random.uniform(-0.1, 0.6)
        self.target_yaw = self.np_random.uniform(-np.pi, np.pi)
        
        self.last_action = np.zeros(2)
        self.last_v = 0.0
        
        self.current_step = 0
        
        return self._get_obs(), {}

    def _get_obs(self):
        pos, orn = p.getBasePositionAndOrientation(self.robotId, physicsClientId=self.client)
        euler = p.getEulerFromQuaternion(orn)
        linear_vel, angular_vel = p.getBaseVelocity(self.robotId, physicsClientId=self.client)
        
        yaw_err = math.sin(self.target_yaw - euler[2])
        v_err = self.target_v - linear_vel[0]
        
        return np.concatenate([
            [euler[1], angular_vel[1], linear_vel[0], v_err, yaw_err, angular_vel[2]],
            self.last_action
        ]).astype(np.float32)

    def step(self, action):
        p.setJointMotorControlArray(self.robotId, [0, 1], p.TORQUE_CONTROL, 
                                    forces=[action[0]*self.max_torque, action[1]*self.max_torque], 
                                    physicsClientId=self.client)
        p.stepSimulation(physicsClientId=self.client)
        
        self.current_step += 1

        obs = self._get_obs()
        pitch, pitch_rate, v_x, v_err, yaw_err = obs[0], obs[1], obs[2], obs[3], obs[4]
        
        v_z = p.getBaseVelocity(self.robotId, physicsClientId=self.client)[0][2]
        
        terminated = abs(pitch) > 0.785
        
        truncated = self.current_step >= self.max_episode_steps

        reward = 10.0 
        reward += 5.0 * math.cos(pitch) 
        reward += 10.0 * math.exp(-(v_err**2) / 0.1) 
        reward += 5.0 * math.exp(-(yaw_err**2) / 0.2) 
        
        reward -= 2.0 * abs(v_z) 
        reward -= 0.5 * np.sum(np.square(action - self.last_action)) 
        reward -= 0.1 * np.sum(np.square(action)) 

        if terminated: 
            reward = -500.0

        self.last_action = action.copy()
        self.last_v = v_x
        
        return obs, reward, terminated, truncated, {}

    def close(self):
        p.disconnect(physicsClientId=self.client)

In [ ]:
'''V7: Endless Gyro'''
#======================================
# Rendered Demonstration
#======================================
try:
    p.disconnect()
except:
    pass

test_env_raw = DummyVecEnv([lambda: BalanceBotEnv(render=True)])

test_env = VecNormalize.load("vec_normalize_v7.pkl", test_env_raw)
test_env.training = False
test_env.norm_reward = False

model = PPO.load("balance_bot_commander_v7")

obs = test_env.reset() 
print("Demo Starts...")

try:
    for i in range(2400):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = test_env.step(action)
        time.sleep(1/240)
except KeyboardInterrupt:
    pass
finally:
    test_env.close()